In [1]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import os
os.environ["OMP_NUM_THREADS"] = "4"

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Regression
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Ensemble
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier

# Clustering
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Neural Network
from sklearn.neural_network import MLPClassifier

#Markdown
from IPython.display import Markdown

In [2]:
# =========================
# 2. LOAD DATA
# =========================
data = pd.read_csv("laptop_preprocessed.csv")

X = data.drop(columns=['Price'])
y_reg = data['Price']

# 🔥 IMPROVED CLASS CREATION (less overlap than qcut)
def categorize_price(p):
    if p < 50000:
        return 0
    elif p < 100000:
        return 1
    else:
        return 2

y_clf = y_reg.apply(categorize_price)

In [3]:
# =========================
# 3. ENCODING + TRAIN TEST SPLIT (FIXED)
# =========================
X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train_reg, y_test_reg, y_train_clf, y_test_clf = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

In [4]:
# =========================
# 4. SCALING (ONLY WHERE NEEDED)
# =========================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
# =========================
# 5. PCA (USED ONLY FOR LINEAR MODELS)
# =========================
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Original Features:", X_train.shape[1])
print("Reduced Features:", X_train_pca.shape[1])

Original Features: 247
Reduced Features: 180


In [6]:
# =========================
# 6. REGRESSION MODELS
# =========================
def evaluate_reg(model, name, X_tr, X_te):
    model.fit(X_tr, y_train_reg)
    y_pred = model.predict(X_te)

    display(Markdown(f'***{name}***'))
    print("MAE :", mean_absolute_error(y_test_reg, y_pred))
    print("MSE :", mean_squared_error(y_test_reg, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test_reg, y_pred)))
    print("R2  :", r2_score(y_test_reg, y_pred))

# 🔥 Better usage: PCA for linear models
evaluate_reg(LinearRegression(), "Linear Regression", X_train_scaled, X_test_scaled)
evaluate_reg(Lasso(alpha=0.1, max_iter=1000), "Lasso Regression (PCA)", X_train_pca, X_test_pca)

***Linear Regression***

MAE : 11566.721471815175
MSE : 341204905.0745022
RMSE: 18471.732595360463
R2  : 0.740934095452666


***Lasso Regression (PCA)***

MAE : 12358.518803678226
MSE : 365831378.41477084
RMSE: 19126.71896627257
R2  : 0.7222360067182311


In [7]:
# =========================
# 7. CLASSIFICATION MODELS
# =========================

def evaluate_clf(model, name, X_tr, X_te):
    model.fit(X_tr, y_train_clf)
    y_pred = model.predict(X_te)

    display(Markdown(f'***{name}***'))    
    print("Accuracy:", accuracy_score(y_test_clf, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test_clf, y_pred))
    print("Classification Report:\n", classification_report(y_test_clf, y_pred))

# 🔥 Tuned models
evaluate_clf(DecisionTreeClassifier(max_depth=10), "Decision Tree", X_train, X_test)
evaluate_clf(KNeighborsClassifier(n_neighbors=7), "KNN", X_train_scaled, X_test_scaled)
evaluate_clf(SVC(kernel='rbf', C=10, gamma='scale'), "SVM (Improved)", X_train_scaled, X_test_scaled)

***Decision Tree***

Accuracy: 0.845771144278607
Confusion Matrix:
 [[130   5   1]
 [ 18  24   4]
 [  1   2  16]]
Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.96      0.91       136
           1       0.77      0.52      0.62        46
           2       0.76      0.84      0.80        19

    accuracy                           0.85       201
   macro avg       0.80      0.77      0.78       201
weighted avg       0.84      0.85      0.84       201



***KNN***

Accuracy: 0.7761194029850746
Confusion Matrix:
 [[128   8   0]
 [ 23  23   0]
 [  5   9   5]]
Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.94      0.88       136
           1       0.57      0.50      0.53        46
           2       1.00      0.26      0.42        19

    accuracy                           0.78       201
   macro avg       0.80      0.57      0.61       201
weighted avg       0.78      0.78      0.75       201



***SVM (Improved)***

Accuracy: 0.845771144278607
Confusion Matrix:
 [[129   7   0]
 [ 16  27   3]
 [  1   4  14]]
Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.95      0.91       136
           1       0.71      0.59      0.64        46
           2       0.82      0.74      0.78        19

    accuracy                           0.85       201
   macro avg       0.81      0.76      0.78       201
weighted avg       0.84      0.85      0.84       201



In [8]:
# =========================
# 8. ENSEMBLE MODELS (IMPROVED)
# =========================
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric='mlogloss',
    random_state=42
)

voting_hard = VotingClassifier(
    estimators=[('rf', rf), ('xgb', xgb)],
    voting='hard'
)

voting_soft = VotingClassifier(
    estimators=[('rf', rf), ('xgb', xgb)],
    voting='soft'
)

# 🔥 Tree models → no scaling needed
evaluate_clf(rf, "Random Forest", X_train, X_test)
evaluate_clf(xgb, "XGBoost", X_train, X_test)
evaluate_clf(voting_hard, "Voting (Hard)", X_train, X_test)
evaluate_clf(voting_soft, "Voting (Soft)", X_train, X_test)

***Random Forest***

Accuracy: 0.8407960199004975
Confusion Matrix:
 [[127   8   1]
 [ 16  26   4]
 [  1   2  16]]
Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.93      0.91       136
           1       0.72      0.57      0.63        46
           2       0.76      0.84      0.80        19

    accuracy                           0.84       201
   macro avg       0.79      0.78      0.78       201
weighted avg       0.83      0.84      0.83       201



***XGBoost***

Accuracy: 0.8308457711442786
Confusion Matrix:
 [[127   8   1]
 [ 18  24   4]
 [  1   2  16]]
Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.93      0.90       136
           1       0.71      0.52      0.60        46
           2       0.76      0.84      0.80        19

    accuracy                           0.83       201
   macro avg       0.78      0.77      0.77       201
weighted avg       0.82      0.83      0.82       201



***Voting (Hard)***

Accuracy: 0.8407960199004975
Confusion Matrix:
 [[127   8   1]
 [ 18  27   1]
 [  1   3  15]]
Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.93      0.90       136
           1       0.71      0.59      0.64        46
           2       0.88      0.79      0.83        19

    accuracy                           0.84       201
   macro avg       0.82      0.77      0.79       201
weighted avg       0.83      0.84      0.84       201



***Voting (Soft)***

Accuracy: 0.8258706467661692
Confusion Matrix:
 [[127   8   1]
 [ 18  22   6]
 [  1   1  17]]
Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.93      0.90       136
           1       0.71      0.48      0.57        46
           2       0.71      0.89      0.79        19

    accuracy                           0.83       201
   macro avg       0.76      0.77      0.75       201
weighted avg       0.82      0.83      0.81       201



In [9]:
display(Markdown('**Clustering Results**'))

# KMeans
display(Markdown('**K-Means**'))
for k in range(2, 5):
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42)
    k_labels = kmeans.fit_predict(X_train_pca)
    display(Markdown(f'**&nbsp;&nbsp;Clusters : {k}**'))
    print("Silhouette Score:", silhouette_score(X_train_pca, k_labels))
    print("Davies-Bouldin Index:", davies_bouldin_score(X_train_pca, k_labels))

# Agglomerative
display(Markdown('**Agglomerative Clustering**'))
for linkage in ['ward', 'complete', 'average']:
    agg = AgglomerativeClustering(n_clusters=3, linkage=linkage)
    a_labels = agg.fit_predict(X_train_pca)
    display(Markdown(f'**&nbsp;&nbsp;linkage = {linkage}**'))
    print("Silhouette Score:", silhouette_score(X_train_pca, a_labels))
    print("Davies-Bouldin Index:", davies_bouldin_score(X_train_pca, a_labels))

**Clustering Results**

**K-Means**

**&nbsp;&nbsp;Clusters : 2**

Silhouette Score: 0.2432792592791203
Davies-Bouldin Index: 0.8346118014673429


**&nbsp;&nbsp;Clusters : 3**

...

Silhouette Score: 0.06788686171202937
Davies-Bouldin Index: 4.192742677127316


**&nbsp;&nbsp;linkage = complete**

Silhouette Score: 0.4442753351099731
Davies-Bouldin Index: 0.8563166308431202


**&nbsp;&nbsp;linkage = average**

Silhouette Score: 0.4953094048272134
Davies-Bouldin Index: 0.3677536575757294


In [10]:
# =========================
# 10. MLP (IMPROVED)
# =========================
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    max_iter=1000,
    learning_rate_init=0.001,
    random_state=42
)

evaluate_clf(mlp, "MLP Classifier", X_train_scaled, X_test_scaled)

***MLP Classifier***

Accuracy: 0.8208955223880597
Confusion Matrix:
 [[120  16   0]
 [ 13  28   5]
 [  0   2  17]]
Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.88      0.89       136
           1       0.61      0.61      0.61        46
           2       0.77      0.89      0.83        19

    accuracy                           0.82       201
   macro avg       0.76      0.80      0.78       201
weighted avg       0.82      0.82      0.82       201

